# 08_06 EES Framework – AHP-Derived EES Score and Sensitivity Analysis

## Purpose

This notebook calculates the final Explainability-Ethics Score (EES) for each model.

It is configurable across German Credit, Pima Diabetes, and COMPAS. Only `ACTIVE_DATASET` should be changed.

## Updated EES Design

The final EES score combines four pillars:

1. Fidelity
2. Stability
3. Actionability
4. Regulatory Alignment

The revised pipeline uses:

```text
Fidelity = 1/3 SHAP Reconstruction Fidelity + 1/3 LIME Local Surrogate Fidelity + 1/3 SHAP-LIME Agreement
```

```text
Actionability = 1/3 Availability + 1/3 Proximity + 1/3 Sparsity
```

## AHP-Derived Weighting Strategy

All weighting schemes in this notebook are derived using the Analytic Hierarchy Process (AHP) eigenvector method.

The primary Master's score is Scheme 1, where all pillars are treated equally through an AHP pairwise matrix of ones.

Sensitivity analysis then evaluates five additional AHP-derived schemes:

| Scheme | Description | Logic |
|---|---|---|
| Scheme 1 | Equal Weight | All pillars equal |
| Scheme 2 | Fidelity Priority | Fidelity moderately preferred over all others |
| Scheme 3 | Stability Priority | Stability moderately preferred over all others |
| Scheme 4 | Actionability Priority | Actionability moderately preferred over all others |
| Scheme 5 | Regulatory Priority | Regulatory Alignment moderately preferred over all others |
| Scheme 6 | Theoretical Priority | Fidelity > Regulatory Alignment > Stability > Actionability |

The primary result remains equal-weight, while the other schemes test ranking robustness.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "results_folder": "german_credit",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_german_credit",
        "stability_folder": "08_03_ees_framework_stability_score_german_credit",
        "actionability_folder": "08_04_ees_framework_actionability_score_german_credit",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_german_credit",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_german_credit"
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "results_folder": "pima_diabetes",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_pima_diabetes",
        "stability_folder": "08_03_ees_framework_stability_score_pima_diabetes",
        "actionability_folder": "08_04_ees_framework_actionability_score_pima_diabetes",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_pima_diabetes",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_pima_diabetes"
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "results_folder": "compas",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_compas",
        "stability_folder": "08_03_ees_framework_stability_score_compas",
        "actionability_folder": "08_04_ees_framework_actionability_score_compas",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_compas",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_compas"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: COMPAS
Dataset name: compas


In [2]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 300)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["ahp_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

FIDELITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["fidelity_folder"]
FIDELITY_TABLES_PATH = FIDELITY_RESULTS_PATH / "tables"
FIDELITY_DATASETS_PATH = FIDELITY_RESULTS_PATH / "datasets"

STABILITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["stability_folder"]
STABILITY_TABLES_PATH = STABILITY_RESULTS_PATH / "tables"
STABILITY_DATASETS_PATH = STABILITY_RESULTS_PATH / "datasets"

ACTIONABILITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["actionability_folder"]
ACTIONABILITY_TABLES_PATH = ACTIONABILITY_RESULTS_PATH / "tables"
ACTIONABILITY_DATASETS_PATH = ACTIONABILITY_RESULTS_PATH / "datasets"

REGULATORY_RESULTS_PATH = RESULTS_ROOT / CONFIG["regulatory_folder"]
REGULATORY_TABLES_PATH = REGULATORY_RESULTS_PATH / "tables"
REGULATORY_DATASETS_PATH = REGULATORY_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("AHP output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/compas
AHP output path: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path

    if required:
        checked = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")

    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def classify_ees_score(score):
    if pd.isna(score):
        return "Not available"
    if score >= 0.80:
        return "High EES"
    if score >= 0.60:
        return "Moderate-high EES"
    if score >= 0.40:
        return "Moderate EES"
    if score >= 0.20:
        return "Low-moderate EES"
    return "Low EES"


def calculate_ahp_weights(pairwise_matrix, labels):
    matrix = np.array(pairwise_matrix, dtype=float)

    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    max_index = np.argmax(eigenvalues.real)

    lambda_max = float(eigenvalues[max_index].real)
    principal_eigenvector = eigenvectors[:, max_index].real

    weights = np.abs(principal_eigenvector)
    weights = weights / weights.sum()

    n = matrix.shape[0]
    consistency_index = (lambda_max - n) / (n - 1) if n > 1 else 0.0

    random_index_lookup = {
        1: 0.00,
        2: 0.00,
        3: 0.58,
        4: 0.90,
        5: 1.12,
        6: 1.24,
        7: 1.32,
        8: 1.41,
        9: 1.45,
        10: 1.49
    }

    random_index = random_index_lookup.get(n, 1.49)
    consistency_ratio = 0.0 if random_index == 0 else consistency_index / random_index

    weight_df = pd.DataFrame({
        "pillar": labels,
        "ahp_weight": weights
    })

    diagnostics = {
        "lambda_max": lambda_max,
        "consistency_index": consistency_index,
        "random_index": random_index,
        "consistency_ratio": consistency_ratio,
        "is_consistent_under_0_10": consistency_ratio <= 0.10
    }

    return weight_df, diagnostics


def create_pairwise_priority_matrix(priority_pillar_index, n_pillars=4, priority_strength=2):
    matrix = np.ones((n_pillars, n_pillars), dtype=float)

    for j in range(n_pillars):
        if j != priority_pillar_index:
            matrix[priority_pillar_index, j] = priority_strength
            matrix[j, priority_pillar_index] = 1 / priority_strength

    return matrix

## 4. Load EES Pillar Scores

In [5]:
# =========================================================
# 4. Load EES Pillar Scores
# =========================================================

fidelity_file = find_first_existing_file(
    [
        FIDELITY_DATASETS_PATH / "ees_fidelity_model_scores.csv",
        FIDELITY_TABLES_PATH / "ees_fidelity_model_scores.csv",
        FIDELITY_DATASETS_PATH / "fidelity_score_summary.csv",
        FIDELITY_TABLES_PATH / "fidelity_score_summary.csv"
    ],
    "Fidelity model scores"
)

stability_file = find_first_existing_file(
    [
        STABILITY_DATASETS_PATH / "ees_stability_model_scores.csv",
        STABILITY_TABLES_PATH / "ees_stability_model_scores.csv",
        STABILITY_DATASETS_PATH / "stability_score_summary.csv",
        STABILITY_TABLES_PATH / "stability_score_summary.csv"
    ],
    "Stability model scores"
)

actionability_file = find_first_existing_file(
    [
        ACTIONABILITY_DATASETS_PATH / "ees_actionability_model_scores.csv",
        ACTIONABILITY_TABLES_PATH / "ees_actionability_model_scores.csv",
        ACTIONABILITY_DATASETS_PATH / "actionability_score_summary.csv",
        ACTIONABILITY_TABLES_PATH / "actionability_score_summary.csv"
    ],
    "Actionability model scores"
)

regulatory_file = find_first_existing_file(
    [
        REGULATORY_DATASETS_PATH / "ees_regulatory_alignment_model_scores.csv",
        REGULATORY_TABLES_PATH / "ees_regulatory_alignment_model_scores.csv",
        REGULATORY_DATASETS_PATH / "regulatory_alignment_score_summary.csv",
        REGULATORY_TABLES_PATH / "regulatory_alignment_score_summary.csv"
    ],
    "Regulatory alignment model scores"
)

fidelity_scores_df = read_csv_safely(fidelity_file, "Fidelity model scores")
stability_scores_df = read_csv_safely(stability_file, "Stability model scores")
actionability_scores_df = read_csv_safely(actionability_file, "Actionability model scores")
regulatory_scores_df = read_csv_safely(regulatory_file, "Regulatory alignment model scores")

print("fidelity_scores_df:", fidelity_scores_df.shape)
print("stability_scores_df:", stability_scores_df.shape)
print("actionability_scores_df:", actionability_scores_df.shape)
print("regulatory_scores_df:", regulatory_scores_df.shape)

display(fidelity_scores_df.head())
display(stability_scores_df.head())
display(actionability_scores_df.head())
display(regulatory_scores_df.head())

Fidelity model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/datasets/ees_fidelity_model_scores.csv
Stability model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/datasets/ees_stability_model_scores.csv
Actionability model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_04_ees_framework_actionability_score_compas/datasets/ees_actionability_model_scores.csv
Regulatory alignment model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/datasets/ees_regulatory_alignment_model_scores.csv
fidelity_scores_df: (5, 18)
stability_scores_df: (5, 15)
actionability_scores_df: (5, 20)
regulatory_scores_df: (5, 14)


,dataset_name,dataset_label,model_id,model_name,shap_reconstruction_fidelity_score,mean_reconstruction_error,shap_reconstruction_available_pct,lime_fidelity_score,mean_lime_r2_score,mean_lime_local_error,shap_lime_agreement_score,mean_kendall_tau,n_fidelity_components_available,fidelity_score,fidelity_score_pct,fidelity_rank,fidelity_band,fidelity_evidence_note
0,compas,COMPAS,xgboost,XGBoost,1.000000,0.000000,100.000000,0.518743,0.518743,0.125687,0.435232,-0.129536,3,0.651325,65.132482,1,Moderate-high fidelity,All three fidelity components available.
1,compas,COMPAS,random_forest,Random Forest,0.999444,0.000556,100.000000,0.404200,0.404200,0.291376,0.439104,-0.121791,3,0.614249,61.424934,2,Moderate-high fidelity,All three fidelity components available.
2,compas,COMPAS,mlp,MLP,1.000000,0.000000,100.000000,0.456811,0.456811,0.138946,0.345068,-0.309864,3,0.600626,60.062642,3,Moderate-high fidelity,All three fidelity components available.
3,compas,COMPAS,decision_tree,Decision Tree,1.000000,0.000000,100.000000,0.263802,0.263802,0.103848,0.511505,0.023010,3,0.591769,59.176894,4,Moderate fidelity,All three fidelity components available.
4,compas,COMPAS,logistic_regression,Logistic Regression,1.000000,0.000000,100.000000,0.366766,0.366766,0.080377,0.336659,-0.326682,3,0.567808,56.780823,5,Moderate fidelity,All three fidelity components available.


,model_id,model_name,n_instances,mean_output_change,median_output_change,max_output_change,mean_perturbation_norm,median_perturbation_norm,mean_local_lipschitz_ratio,median_local_lipschitz_ratio,max_local_lipschitz_ratio,stability_score,stability_score_pct,stability_rank,stability_band
0,decision_tree,Decision Tree,3500,0.008595,0.000000,0.810897,0.271562,0.270657,0.031236,0.000000,3.610268,1.000000,100.000000,1,High stability
1,logistic_regression,Logistic Regression,3500,0.035847,0.031113,0.219134,0.271717,0.270979,0.132007,0.116251,0.713597,0.661564,66.156446,2,Moderate-high stability
2,mlp,MLP,3500,0.038214,0.032456,0.283023,0.271664,0.270831,0.140674,0.120986,0.934986,0.632455,63.245514,3,Moderate-high stability
3,xgboost,XGBoost,3500,0.038602,0.026659,0.613785,0.271551,0.270658,0.144497,0.098853,2.787682,0.619616,61.961607,4,Moderate-high stability
4,random_forest,Random Forest,3500,0.089450,0.017688,1.000000,0.271612,0.270508,0.328991,0.063693,4.907182,0.000000,0.000000,5,Low stability


,dataset_name,dataset_label,model_id,model_name,availability_score,proximity_score,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band,score_source,n_unfavourable_instances,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,median_min_normalized_numeric_distance,mean_changed_features,median_changed_features,n_skipped_due_to_missing_values,total_features_for_sparsity
0,compas,COMPAS,decision_tree,Decision Tree,1.000000,0.959375,0.892237,0.950537,95.053733,1,High actionability,recalculated_from_instance_level_counterfactuals,1752,1752,0.040625,0.034039,1.077626,1.000000,0,10
1,compas,COMPAS,logistic_regression,Logistic Regression,1.000000,0.958794,0.887626,0.948806,94.880645,2,High actionability,recalculated_from_instance_level_counterfactuals,1592,1592,0.041206,0.036163,1.123744,1.000000,0,10
2,compas,COMPAS,xgboost,XGBoost,1.000000,0.955618,0.882705,0.946108,94.610756,3,High actionability,recalculated_from_instance_level_counterfactuals,1671,1671,0.044382,0.035680,1.172950,1.000000,0,10
3,compas,COMPAS,mlp,MLP,1.000000,0.950526,0.881861,0.944129,94.412914,4,High actionability,recalculated_from_instance_level_counterfactuals,1698,1698,0.049474,0.040370,1.181390,1.000000,0,10
4,compas,COMPAS,random_forest,Random Forest,1.000000,0.956268,0.875014,0.943761,94.376091,5,High actionability,recalculated_from_instance_level_counterfactuals,1741,1741,0.043732,0.035714,1.249856,1.000000,0,10


,dataset_name,dataset_label,model_id,model_name,transparency_score,explanation_availability_score,sensitive_feature_awareness_score,documentation_auditability_score,criteria_satisfied,criteria_total,regulatory_alignment_score,regulatory_alignment_score_pct,evidence_status,regulatory_alignment_rank
0,compas,COMPAS,decision_tree,Decision Tree,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
1,compas,COMPAS,logistic_regression,Logistic Regression,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
2,compas,COMPAS,mlp,MLP,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
3,compas,COMPAS,random_forest,Random Forest,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
4,compas,COMPAS,xgboost,XGBoost,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1


## 5. Validate Required Pillar Columns

In [6]:
# =========================================================
# 5. Validate Required Pillar Columns
# =========================================================

validation_rows = []

required_tables = [
    ("fidelity", fidelity_scores_df, ["model_id", "model_name", "fidelity_score"]),
    ("stability", stability_scores_df, ["model_id", "model_name", "stability_score"]),
    ("actionability", actionability_scores_df, ["model_id", "model_name", "actionability_score"]),
    ("regulatory", regulatory_scores_df, ["model_id", "model_name", "regulatory_alignment_score"]),
]

for table_name, df, required_cols in required_tables:
    validation_rows.append({
        "check": f"{table_name} table not empty",
        "passed": not df.empty
    })

    for col in required_cols:
        validation_rows.append({
            "check": f"{table_name} has {col}",
            "passed": col in df.columns
        })

ees_ahp_input_validation_df = pd.DataFrame(validation_rows)

display(ees_ahp_input_validation_df)

save_df(ees_ahp_input_validation_df, "ees_ahp_input_validation.csv")

if not ees_ahp_input_validation_df["passed"].all():
    raise ValueError("One or more EES AHP input validation checks failed.")

,check,passed
0,fidelity table not empty,True
1,fidelity has model_id,True
2,fidelity has model_name,True
3,fidelity has fidelity_score,True
4,stability table not empty,True
5,stability has model_id,True
6,stability has model_name,True
7,stability has stability_score,True
8,actionability table not empty,True
9,actionability has model_id,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_input_validation.csv


## 6. Merge Pillar Scores

In [7]:
# =========================================================
# 6. Merge Pillar Scores
# =========================================================

ees_component_scores_df = (
    fidelity_scores_df[["model_id", "model_name", "fidelity_score"]]
    .merge(
        stability_scores_df[["model_id", "stability_score"]],
        on="model_id",
        how="outer"
    )
    .merge(
        actionability_scores_df[["model_id", "actionability_score"]],
        on="model_id",
        how="outer"
    )
    .merge(
        regulatory_scores_df[["model_id", "regulatory_alignment_score"]],
        on="model_id",
        how="outer"
    )
)

if ees_component_scores_df["model_name"].isna().any():
    model_name_lookup = pd.concat(
        [
            fidelity_scores_df[["model_id", "model_name"]],
            stability_scores_df[["model_id", "model_name"]],
            actionability_scores_df[["model_id", "model_name"]],
            regulatory_scores_df[["model_id", "model_name"]]
        ],
        ignore_index=True
    ).drop_duplicates("model_id")

    ees_component_scores_df = ees_component_scores_df.drop(columns=["model_name"], errors="ignore")
    ees_component_scores_df = ees_component_scores_df.merge(
        model_name_lookup,
        on="model_id",
        how="left"
    )

pillar_columns = [
    "fidelity_score",
    "stability_score",
    "actionability_score",
    "regulatory_alignment_score"
]

pillar_labels = [
    "Fidelity",
    "Stability",
    "Actionability",
    "Regulatory Alignment"
]

ees_component_scores_df["dataset_name"] = DATASET_NAME
ees_component_scores_df["dataset_label"] = CONFIG["dataset_label"]
ees_component_scores_df["n_components_available"] = ees_component_scores_df[pillar_columns].notna().sum(axis=1)

display(ees_component_scores_df)
save_df(ees_component_scores_df, "ees_component_scores_input.csv")

,model_id,model_name,fidelity_score,stability_score,actionability_score,regulatory_alignment_score,dataset_name,dataset_label,n_components_available
0,decision_tree,Decision Tree,0.591769,1.000000,0.950537,1.000000,compas,COMPAS,4
1,logistic_regression,Logistic Regression,0.567808,0.661564,0.948806,1.000000,compas,COMPAS,4
2,mlp,MLP,0.600626,0.632455,0.944129,1.000000,compas,COMPAS,4
3,random_forest,Random Forest,0.614249,0.000000,0.943761,1.000000,compas,COMPAS,4
4,xgboost,XGBoost,0.651325,0.619616,0.946108,1.000000,compas,COMPAS,4


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_component_scores_input.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_component_scores_input.csv')

## 7. Define AHP-Derived Weighting Schemes

All schemes are represented with AHP pairwise matrices.

In [8]:
# =========================================================
# 7. Define AHP-Derived Weighting Schemes
# =========================================================

scheme_definitions = {
    "scheme_1_equal_weight": {
        "scheme_id": "scheme_1_equal_weight",
        "scheme_name": "Scheme 1 - Equal Weight",
        "scheme_role": "Primary Master's EES score",
        "description": "All EES pillars are equally important.",
        "pairwise_matrix": np.ones((4, 4), dtype=float)
    },
    "scheme_2_fidelity_priority": {
        "scheme_id": "scheme_2_fidelity_priority",
        "scheme_name": "Scheme 2 - Fidelity Priority",
        "scheme_role": "Sensitivity analysis",
        "description": "Fidelity is moderately preferred over all other pillars.",
        "pairwise_matrix": create_pairwise_priority_matrix(0, 4, 2)
    },
    "scheme_3_stability_priority": {
        "scheme_id": "scheme_3_stability_priority",
        "scheme_name": "Scheme 3 - Stability Priority",
        "scheme_role": "Sensitivity analysis",
        "description": "Stability is moderately preferred over all other pillars.",
        "pairwise_matrix": create_pairwise_priority_matrix(1, 4, 2)
    },
    "scheme_4_actionability_priority": {
        "scheme_id": "scheme_4_actionability_priority",
        "scheme_name": "Scheme 4 - Actionability Priority",
        "scheme_role": "Sensitivity analysis",
        "description": "Actionability is moderately preferred over all other pillars.",
        "pairwise_matrix": create_pairwise_priority_matrix(2, 4, 2)
    },
    "scheme_5_regulatory_priority": {
        "scheme_id": "scheme_5_regulatory_priority",
        "scheme_name": "Scheme 5 - Regulatory Alignment Priority",
        "scheme_role": "Sensitivity analysis",
        "description": "Regulatory Alignment is moderately preferred over all other pillars.",
        "pairwise_matrix": create_pairwise_priority_matrix(3, 4, 2)
    },
    "scheme_6_theoretical_priority": {
        "scheme_id": "scheme_6_theoretical_priority",
        "scheme_name": "Scheme 6 - Theoretical Priority F > R > S > A",
        "scheme_role": "Sensitivity analysis",
        "description": "Theoretical ordering: Fidelity > Regulatory Alignment > Stability > Actionability.",
        "pairwise_matrix": np.array([
            [1,     3,     4,     2],
            [1/3,   1,     2,     1/2],
            [1/4,   1/2,   1,     1/3],
            [1/2,   2,     3,     1]
        ], dtype=float)
    }
}

scheme_metadata_rows = []
scheme_weight_rows = []
scheme_diagnostic_rows = []
scheme_matrix_rows = []

for scheme_id, scheme_info in scheme_definitions.items():

    pairwise_matrix = scheme_info["pairwise_matrix"]

    weight_df, diagnostics = calculate_ahp_weights(
        pairwise_matrix=pairwise_matrix,
        labels=pillar_labels
    )

    for i, pillar in enumerate(pillar_labels):
        scheme_weight_rows.append({
            "scheme_id": scheme_id,
            "scheme_name": scheme_info["scheme_name"],
            "scheme_role": scheme_info["scheme_role"],
            "pillar": pillar,
            "pillar_column": pillar_columns[i],
            "ahp_weight": weight_df.loc[i, "ahp_weight"],
            "ahp_weight_pct": weight_df.loc[i, "ahp_weight"] * 100
        })

    scheme_diagnostic_rows.append({
        "scheme_id": scheme_id,
        "scheme_name": scheme_info["scheme_name"],
        "scheme_role": scheme_info["scheme_role"],
        **diagnostics
    })

    scheme_metadata_rows.append({
        "scheme_id": scheme_id,
        "scheme_name": scheme_info["scheme_name"],
        "scheme_role": scheme_info["scheme_role"],
        "description": scheme_info["description"]
    })

    matrix_df = pd.DataFrame(
        pairwise_matrix,
        columns=pillar_labels,
        index=pillar_labels
    ).reset_index().rename(columns={"index": "row_pillar"})

    matrix_df.insert(0, "scheme_id", scheme_id)
    matrix_df.insert(1, "scheme_name", scheme_info["scheme_name"])

    scheme_matrix_rows.append(matrix_df)

ahp_scheme_metadata_df = pd.DataFrame(scheme_metadata_rows)
ahp_scheme_weights_df = pd.DataFrame(scheme_weight_rows)
ahp_scheme_diagnostics_df = pd.DataFrame(scheme_diagnostic_rows)
ahp_pairwise_matrices_df = pd.concat(scheme_matrix_rows, ignore_index=True)

display(ahp_scheme_metadata_df)
display(ahp_scheme_weights_df)
display(ahp_scheme_diagnostics_df)
display(ahp_pairwise_matrices_df)

save_df(ahp_scheme_metadata_df, "ees_ahp_scheme_metadata.csv")
save_df(ahp_scheme_weights_df, "ees_ahp_scheme_weights.csv")
save_df(ahp_scheme_diagnostics_df, "ees_ahp_scheme_consistency_diagnostics.csv")
save_df(ahp_pairwise_matrices_df, "ees_ahp_pairwise_matrices.csv")

,scheme_id,scheme_name,scheme_role,description
0,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,All EES pillars are equally important.
1,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Fidelity is moderately preferred over all othe...
2,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Stability is moderately preferred over all oth...
3,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,Actionability is moderately preferred over all...
4,scheme_5_regulatory_priority,Scheme 5 - Regulatory Alignment Priority,Sensitivity analysis,Regulatory Alignment is moderately preferred o...
5,scheme_6_theoretical_priority,Scheme 6 - Theoretical Priority F > R > S > A,Sensitivity analysis,Theoretical ordering: Fidelity > Regulatory Al...


,scheme_id,scheme_name,scheme_role,pillar,pillar_column,ahp_weight,ahp_weight_pct
0,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Fidelity,fidelity_score,0.250000,25.000000
1,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Stability,stability_score,0.250000,25.000000
2,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Actionability,actionability_score,0.250000,25.000000
3,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Regulatory Alignment,regulatory_alignment_score,0.250000,25.000000
4,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Fidelity,fidelity_score,0.400000,40.000000
5,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Stability,stability_score,0.200000,20.000000
6,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Actionability,actionability_score,0.200000,20.000000
7,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Regulatory Alignment,regulatory_alignment_score,0.200000,20.000000
8,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Fidelity,fidelity_score,0.200000,20.000000
9,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Stability,stability_score,0.400000,40.000000


,scheme_id,scheme_name,scheme_role,lambda_max,consistency_index,random_index,consistency_ratio,is_consistent_under_0_10
0,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,4.000000,-0.000000,0.900000,-0.000000,True
1,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
2,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
3,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
4,scheme_5_regulatory_priority,Scheme 5 - Regulatory Alignment Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
5,scheme_6_theoretical_priority,Scheme 6 - Theoretical Priority F > R > S > A,Sensitivity analysis,4.030983,0.010328,0.900000,0.011475,True


,scheme_id,scheme_name,row_pillar,Fidelity,Stability,Actionability,Regulatory Alignment
0,scheme_1_equal_weight,Scheme 1 - Equal Weight,Fidelity,1.000000,1.000000,1.000000,1.000000
1,scheme_1_equal_weight,Scheme 1 - Equal Weight,Stability,1.000000,1.000000,1.000000,1.000000
2,scheme_1_equal_weight,Scheme 1 - Equal Weight,Actionability,1.000000,1.000000,1.000000,1.000000
3,scheme_1_equal_weight,Scheme 1 - Equal Weight,Regulatory Alignment,1.000000,1.000000,1.000000,1.000000
4,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Fidelity,1.000000,2.000000,2.000000,2.000000
5,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Stability,0.500000,1.000000,1.000000,1.000000
6,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Actionability,0.500000,1.000000,1.000000,1.000000
7,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Regulatory Alignment,0.500000,1.000000,1.000000,1.000000
8,scheme_3_stability_priority,Scheme 3 - Stability Priority,Fidelity,1.000000,0.500000,1.000000,1.000000
9,scheme_3_stability_priority,Scheme 3 - Stability Priority,Stability,2.000000,1.000000,2.000000,2.000000


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_scheme_metadata.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_scheme_weights.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_scheme_consistency_diagnostics.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_pairwise_matrices.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_pairwise_matrices.csv')

## 8. Calculate EES Scores Under All AHP Schemes

In [9]:
# =========================================================
# 8. Calculate EES Scores Under All AHP Schemes
# =========================================================

ees_scores_all_schemes_df = ees_component_scores_df.copy()

for scheme_id in scheme_definitions.keys():

    scheme_weights_df = ahp_scheme_weights_df[ahp_scheme_weights_df["scheme_id"] == scheme_id].copy()
    scheme_weight_map = dict(zip(scheme_weights_df["pillar_column"], scheme_weights_df["ahp_weight"]))

    score_col = f"{scheme_id}_score"
    pct_col = f"{scheme_id}_score_pct"
    rank_col = f"{scheme_id}_rank"

    ees_scores_all_schemes_df[score_col] = sum(
        ees_scores_all_schemes_df[col] * scheme_weight_map[col]
        for col in pillar_columns
    )

    ees_scores_all_schemes_df[pct_col] = ees_scores_all_schemes_df[score_col] * 100
    ees_scores_all_schemes_df[rank_col] = (
        ees_scores_all_schemes_df[score_col]
        .rank(ascending=False, method="dense")
        .astype(int)
    )

# Primary score remains AHP-derived equal-weight Scheme 1.
ees_scores_all_schemes_df["ees_score"] = ees_scores_all_schemes_df["scheme_1_equal_weight_score"]
ees_scores_all_schemes_df["ees_score_pct"] = ees_scores_all_schemes_df["scheme_1_equal_weight_score_pct"]
ees_scores_all_schemes_df["ees_rank"] = ees_scores_all_schemes_df["scheme_1_equal_weight_rank"]
ees_scores_all_schemes_df["ees_band"] = ees_scores_all_schemes_df["ees_score"].apply(classify_ees_score)

# Compatibility columns for existing consolidation notebooks.
ees_scores_all_schemes_df["ees_equal_weight_score"] = ees_scores_all_schemes_df["scheme_1_equal_weight_score"]
ees_scores_all_schemes_df["ees_equal_weight_score_pct"] = ees_scores_all_schemes_df["scheme_1_equal_weight_score_pct"]
ees_scores_all_schemes_df["ees_ahp_score"] = ees_scores_all_schemes_df["scheme_6_theoretical_priority_score"]
ees_scores_all_schemes_df["ees_ahp_score_pct"] = ees_scores_all_schemes_df["scheme_6_theoretical_priority_score_pct"]

ees_scores_all_schemes_df = (
    ees_scores_all_schemes_df
    .sort_values(["ees_rank", "model_id"])
    .reset_index(drop=True)
)

display(ees_scores_all_schemes_df)

,model_id,model_name,fidelity_score,stability_score,actionability_score,regulatory_alignment_score,dataset_name,dataset_label,n_components_available,scheme_1_equal_weight_score,scheme_1_equal_weight_score_pct,scheme_1_equal_weight_rank,scheme_2_fidelity_priority_score,scheme_2_fidelity_priority_score_pct,scheme_2_fidelity_priority_rank,scheme_3_stability_priority_score,scheme_3_stability_priority_score_pct,scheme_3_stability_priority_rank,scheme_4_actionability_priority_score,scheme_4_actionability_priority_score_pct,scheme_4_actionability_priority_rank,scheme_5_regulatory_priority_score,scheme_5_regulatory_priority_score_pct,scheme_5_regulatory_priority_rank,scheme_6_theoretical_priority_score,scheme_6_theoretical_priority_score_pct,scheme_6_theoretical_priority_rank,ees_score,ees_score_pct,ees_rank,ees_band,ees_equal_weight_score,ees_equal_weight_score_pct,ees_ahp_score,ees_ahp_score_pct
0,decision_tree,Decision Tree,0.591769,1.000000,0.950537,1.000000,compas,COMPAS,4,0.885577,88.557657,1,0.826815,82.681504,1,0.908461,90.846125,1,0.898569,89.856872,1,0.908461,90.846125,1,0.804515,80.451480,1,0.885577,88.557657,1,High EES,0.885577,88.557657,0.804515,80.451480
1,xgboost,XGBoost,0.651325,0.619616,0.946108,1.000000,compas,COMPAS,4,0.804262,80.426211,2,0.773675,77.367465,2,0.767333,76.733290,3,0.832631,83.263120,2,0.843410,84.340969,2,0.771027,77.102718,2,0.804262,80.426211,2,High EES,0.804262,80.426211,0.771027,77.102718
2,logistic_regression,Logistic Regression,0.567808,0.661564,0.948806,1.000000,compas,COMPAS,4,0.794545,79.454478,3,0.749197,74.919747,4,0.767949,76.794872,2,0.825397,82.539712,3,0.835636,83.563583,3,0.738973,73.897324,4,0.794545,79.454478,3,Moderate-high EES,0.794545,79.454478,0.738973,73.897324
3,mlp,MLP,0.600626,0.632455,0.944129,1.000000,compas,COMPAS,4,0.794303,79.430268,4,0.755567,75.556743,3,0.761933,76.193317,4,0.824268,82.426797,4,0.835442,83.544214,4,0.749203,74.920260,3,0.794303,79.430268,4,Moderate-high EES,0.794303,79.430268,0.749203,74.920260
4,random_forest,Random Forest,0.614249,0.000000,0.943761,1.000000,compas,COMPAS,4,0.639503,63.950256,5,0.634452,63.445192,5,0.511602,51.160205,5,0.700354,70.035423,5,0.711602,71.160205,5,0.654285,65.428462,5,0.639503,63.950256,5,Moderate-high EES,0.639503,63.950256,0.654285,65.428462


## 9. Create Long-Format Sensitivity Results

In [10]:
# =========================================================
# 9. Create Long-Format Sensitivity Results
# =========================================================

sensitivity_rows = []

for _, model_row in ees_component_scores_df.iterrows():

    for scheme_id, scheme_info in scheme_definitions.items():

        scheme_weights_df = ahp_scheme_weights_df[ahp_scheme_weights_df["scheme_id"] == scheme_id].copy()
        scheme_weight_map = dict(zip(scheme_weights_df["pillar_column"], scheme_weights_df["ahp_weight"]))

        scheme_score = sum(
            model_row[pillar_col] * scheme_weight_map[pillar_col]
            for pillar_col in pillar_columns
        )

        sensitivity_rows.append({
            "dataset_name": DATASET_NAME,
            "dataset_label": CONFIG["dataset_label"],
            "model_id": model_row["model_id"],
            "model_name": model_row["model_name"],
            "scheme_id": scheme_id,
            "scheme_name": scheme_info["scheme_name"],
            "scheme_role": scheme_info["scheme_role"],
            "ees_score_under_scheme": scheme_score,
            "ees_score_under_scheme_pct": scheme_score * 100
        })

ees_weighting_sensitivity_long_df = pd.DataFrame(sensitivity_rows)

ees_weighting_sensitivity_long_df["rank_under_scheme"] = (
    ees_weighting_sensitivity_long_df
    .groupby("scheme_id")["ees_score_under_scheme"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

scheme_1_ranks = (
    ees_weighting_sensitivity_long_df[ees_weighting_sensitivity_long_df["scheme_id"] == "scheme_1_equal_weight"]
    [["model_id", "rank_under_scheme"]]
    .rename(columns={"rank_under_scheme": "scheme_1_equal_weight_rank"})
)

ees_weighting_sensitivity_long_df = ees_weighting_sensitivity_long_df.merge(
    scheme_1_ranks,
    on="model_id",
    how="left"
)

ees_weighting_sensitivity_long_df["absolute_rank_change_vs_equal_weight"] = (
    ees_weighting_sensitivity_long_df["rank_under_scheme"] -
    ees_weighting_sensitivity_long_df["scheme_1_equal_weight_rank"]
).abs()

display(ees_weighting_sensitivity_long_df.head(30))

,dataset_name,dataset_label,model_id,model_name,scheme_id,scheme_name,scheme_role,ees_score_under_scheme,ees_score_under_scheme_pct,rank_under_scheme,scheme_1_equal_weight_rank,absolute_rank_change_vs_equal_weight
0,compas,COMPAS,decision_tree,Decision Tree,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,0.885577,88.557657,1,1,0
1,compas,COMPAS,decision_tree,Decision Tree,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,0.826815,82.681504,1,1,0
2,compas,COMPAS,decision_tree,Decision Tree,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,0.908461,90.846125,1,1,0
3,compas,COMPAS,decision_tree,Decision Tree,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,0.898569,89.856872,1,1,0
4,compas,COMPAS,decision_tree,Decision Tree,scheme_5_regulatory_priority,Scheme 5 - Regulatory Alignment Priority,Sensitivity analysis,0.908461,90.846125,1,1,0
5,compas,COMPAS,decision_tree,Decision Tree,scheme_6_theoretical_priority,Scheme 6 - Theoretical Priority F > R > S > A,Sensitivity analysis,0.804515,80.451480,1,1,0
6,compas,COMPAS,logistic_regression,Logistic Regression,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,0.794545,79.454478,3,3,0
7,compas,COMPAS,logistic_regression,Logistic Regression,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,0.749197,74.919747,4,3,1
8,compas,COMPAS,logistic_regression,Logistic Regression,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,0.767949,76.794872,2,3,1
9,compas,COMPAS,logistic_regression,Logistic Regression,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,0.825397,82.539712,3,3,0


## 10. Create Ranking Robustness Summary

In [11]:
# =========================================================
# 10. Create Ranking Robustness Summary
# =========================================================

ranking_robustness_summary_df = (
    ees_weighting_sensitivity_long_df
    .groupby(["model_id", "model_name"], as_index=False)
    .agg(
        mean_rank_across_schemes=("rank_under_scheme", "mean"),
        median_rank_across_schemes=("rank_under_scheme", "median"),
        min_rank_across_schemes=("rank_under_scheme", "min"),
        max_rank_across_schemes=("rank_under_scheme", "max"),
        rank_range_across_schemes=("rank_under_scheme", lambda x: int(x.max() - x.min())),
        mean_score_across_schemes=("ees_score_under_scheme", "mean"),
        min_score_across_schemes=("ees_score_under_scheme", "min"),
        max_score_across_schemes=("ees_score_under_scheme", "max"),
        max_rank_change_vs_equal_weight=("absolute_rank_change_vs_equal_weight", "max"),
        mean_rank_change_vs_equal_weight=("absolute_rank_change_vs_equal_weight", "mean")
    )
)

ranking_robustness_summary_df["ranking_robustness_note"] = np.where(
    ranking_robustness_summary_df["max_rank_change_vs_equal_weight"] == 0,
    "No rank change across sensitivity schemes.",
    np.where(
        ranking_robustness_summary_df["max_rank_change_vs_equal_weight"] <= 1,
        "Minor rank movement across sensitivity schemes.",
        "Material rank movement across sensitivity schemes."
    )
)

ranking_robustness_summary_df = (
    ranking_robustness_summary_df
    .sort_values(["mean_rank_across_schemes", "model_id"])
    .reset_index(drop=True)
)

display(ranking_robustness_summary_df)

,model_id,model_name,mean_rank_across_schemes,median_rank_across_schemes,min_rank_across_schemes,max_rank_across_schemes,rank_range_across_schemes,mean_score_across_schemes,min_score_across_schemes,max_score_across_schemes,max_rank_change_vs_equal_weight,mean_rank_change_vs_equal_weight,ranking_robustness_note
0,decision_tree,Decision Tree,1.000000,1.000000,1,1,0,0.872066,0.804515,0.908461,0,0.000000,No rank change across sensitivity schemes.
1,xgboost,XGBoost,2.166667,2.000000,2,3,1,0.798723,0.767333,0.843410,1,0.166667,Minor rank movement across sensitivity schemes.
2,logistic_regression,Logistic Regression,3.166667,3.000000,2,4,2,0.785283,0.738973,0.835636,1,0.500000,Minor rank movement across sensitivity schemes.
3,mlp,MLP,3.666667,4.000000,3,4,1,0.786786,0.749203,0.835442,1,0.333333,Minor rank movement across sensitivity schemes.
4,random_forest,Random Forest,5.000000,5.000000,5,5,0,0.641966,0.511602,0.711602,0,0.000000,No rank change across sensitivity schemes.


## 11. Component Contribution Under Primary Scheme

In [12]:
# =========================================================
# 11. Component Contribution Under Primary Scheme
# =========================================================

primary_scheme_id = "scheme_1_equal_weight"
primary_weights_df = ahp_scheme_weights_df[ahp_scheme_weights_df["scheme_id"] == primary_scheme_id].copy()
primary_weight_map = dict(zip(primary_weights_df["pillar_column"], primary_weights_df["ahp_weight"]))

component_long_rows = []

for _, row in ees_scores_all_schemes_df.iterrows():
    for pillar_col, pillar_label in zip(pillar_columns, pillar_labels):
        component_score = row[pillar_col]
        component_weight = primary_weight_map[pillar_col]
        weighted_contribution = component_score * component_weight

        component_long_rows.append({
            "dataset_name": DATASET_NAME,
            "dataset_label": CONFIG["dataset_label"],
            "model_id": row["model_id"],
            "model_name": row["model_name"],
            "pillar": pillar_label,
            "pillar_column": pillar_col,
            "pillar_score": component_score,
            "primary_ahp_weight": component_weight,
            "weighted_contribution": weighted_contribution
        })

ees_component_contribution_long_df = pd.DataFrame(component_long_rows)

display(ees_component_contribution_long_df.head(20))

,dataset_name,dataset_label,model_id,model_name,pillar,pillar_column,pillar_score,primary_ahp_weight,weighted_contribution
0,compas,COMPAS,decision_tree,Decision Tree,Fidelity,fidelity_score,0.591769,0.250000,0.147942
1,compas,COMPAS,decision_tree,Decision Tree,Stability,stability_score,1.000000,0.250000,0.250000
2,compas,COMPAS,decision_tree,Decision Tree,Actionability,actionability_score,0.950537,0.250000,0.237634
3,compas,COMPAS,decision_tree,Decision Tree,Regulatory Alignment,regulatory_alignment_score,1.000000,0.250000,0.250000
4,compas,COMPAS,xgboost,XGBoost,Fidelity,fidelity_score,0.651325,0.250000,0.162831
5,compas,COMPAS,xgboost,XGBoost,Stability,stability_score,0.619616,0.250000,0.154904
6,compas,COMPAS,xgboost,XGBoost,Actionability,actionability_score,0.946108,0.250000,0.236527
7,compas,COMPAS,xgboost,XGBoost,Regulatory Alignment,regulatory_alignment_score,1.000000,0.250000,0.250000
8,compas,COMPAS,logistic_regression,Logistic Regression,Fidelity,fidelity_score,0.567808,0.250000,0.141952
9,compas,COMPAS,logistic_regression,Logistic Regression,Stability,stability_score,0.661564,0.250000,0.165391


## 12. Interpretation Table

In [13]:
# =========================================================
# 12. Interpretation Table
# =========================================================

ees_interpretation_df = pd.DataFrame([
    {"score_range": "0.80 to 1.00", "band": "High EES", "interpretation": "Strong explainability-ethics performance across EES pillars."},
    {"score_range": "0.60 to <0.80", "band": "Moderate-high EES", "interpretation": "Good explainability-ethics performance with some improvement areas."},
    {"score_range": "0.40 to <0.60", "band": "Moderate EES", "interpretation": "Mixed performance across EES pillars."},
    {"score_range": "0.20 to <0.40", "band": "Low-moderate EES", "interpretation": "Limited explainability-ethics performance."},
    {"score_range": "0.00 to <0.20", "band": "Low EES", "interpretation": "Weak explainability-ethics performance across most pillars."}
])

display(ees_interpretation_df)
save_df(ees_interpretation_df, "ees_score_interpretation.csv")

,score_range,band,interpretation
0,0.80 to 1.00,High EES,Strong explainability-ethics performance acros...
1,0.60 to <0.80,Moderate-high EES,Good explainability-ethics performance with so...
2,0.40 to <0.60,Moderate EES,Mixed performance across EES pillars.
3,0.20 to <0.40,Low-moderate EES,Limited explainability-ethics performance.
4,0.00 to <0.20,Low EES,Weak explainability-ethics performance across ...


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_score_interpretation.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_score_interpretation.csv')

## 13. Save AHP / EES Outputs

In [14]:
# =========================================================
# 13. Save AHP / EES Outputs
# =========================================================

save_df(ees_scores_all_schemes_df, "ees_ahp_score_summary.csv")
save_df(ees_scores_all_schemes_df, "ees_final_model_scores.csv")
save_df(ees_component_scores_df, "ees_component_scores_input.csv")
save_df(ees_component_contribution_long_df, "ees_component_contribution_long.csv")
save_df(ees_weighting_sensitivity_long_df, "ees_weighting_sensitivity.csv")
save_df(ees_weighting_sensitivity_long_df, "ees_weighting_sensitivity_long.csv")
save_df(ranking_robustness_summary_df, "ees_ranking_robustness_summary.csv")
save_df(ahp_scheme_weights_df, "ees_ahp_scheme_weights.csv")
save_df(ahp_scheme_diagnostics_df, "ees_ahp_scheme_consistency_diagnostics.csv")
save_df(ahp_pairwise_matrices_df, "ees_ahp_pairwise_matrices.csv")
save_df(ahp_scheme_metadata_df, "ees_ahp_scheme_metadata.csv")

save_df(ees_scores_all_schemes_df, "ees_ahp_score_summary.csv", folder=DATASETS_PATH)
save_df(ees_scores_all_schemes_df, "ees_final_model_scores.csv", folder=DATASETS_PATH)
save_df(ees_component_scores_df, "ees_component_scores_input.csv", folder=DATASETS_PATH)
save_df(ees_component_contribution_long_df, "ees_component_contribution_long.csv", folder=DATASETS_PATH)
save_df(ees_weighting_sensitivity_long_df, "ees_weighting_sensitivity.csv", folder=DATASETS_PATH)
save_df(ees_weighting_sensitivity_long_df, "ees_weighting_sensitivity_long.csv", folder=DATASETS_PATH)
save_df(ranking_robustness_summary_df, "ees_ranking_robustness_summary.csv", folder=DATASETS_PATH)
save_df(ahp_scheme_weights_df, "ees_ahp_scheme_weights.csv", folder=DATASETS_PATH)
save_df(ahp_scheme_diagnostics_df, "ees_ahp_scheme_consistency_diagnostics.csv", folder=DATASETS_PATH)
save_df(ahp_pairwise_matrices_df, "ees_ahp_pairwise_matrices.csv", folder=DATASETS_PATH)
save_df(ahp_scheme_metadata_df, "ees_ahp_scheme_metadata.csv", folder=DATASETS_PATH)

save_excel(
    {
        "ees_final_scores": ees_scores_all_schemes_df,
        "component_scores": ees_component_scores_df,
        "component_contributions": ees_component_contribution_long_df,
        "sensitivity_long": ees_weighting_sensitivity_long_df,
        "ranking_robustness": ranking_robustness_summary_df,
        "scheme_weights": ahp_scheme_weights_df,
        "scheme_diagnostics": ahp_scheme_diagnostics_df,
        "pairwise_matrices": ahp_pairwise_matrices_df,
        "scheme_metadata": ahp_scheme_metadata_df,
        "interpretation": ees_interpretation_df,
        "input_validation": ees_ahp_input_validation_df
    },
    f"ees_ahp_score_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "primary_score": "scheme_1_equal_weight_score",
    "primary_score_rationale": "AHP-derived equal weighting is used as the primary Master's score because no universally accepted weighting hierarchy exists across EES pillars.",
    "sensitivity_schemes": list(scheme_definitions.keys()),
    "theoretical_scheme": "scheme_6_theoretical_priority",
    "theoretical_scheme_ordering": "Fidelity > Regulatory Alignment > Stability > Actionability",
    "updated_fidelity_design": "SHAP reconstruction, LIME local surrogate fidelity and SHAP-LIME agreement.",
    "updated_actionability_design": "Availability, numeric proximity and sparsity.",
    "standardisation_note": "This notebook uses AHP-derived weights for both the primary equal-weight score and all sensitivity schemes."
}

with open(LOGS_PATH / "ees_ahp_score_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "ees_ahp_score_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All EES AHP outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ahp_score_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_final_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_component_scores_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_component_contribution_long.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_weighting_sensitivity.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_weighting_sensitivity_long.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables/ees_ranking_robustness_summary.

## 14. Interpretation Note

The final EES score summarises each model's explainability-ethics quality across four pillars.

The primary reported score is the equal-weight AHP-derived score because it is transparent, conservative, and avoids unsupported subjective prioritisation.

The additional AHP-derived sensitivity schemes evaluate whether model rankings remain stable under alternative weighting assumptions, including the original theoretical ordering:

```text
Fidelity > Regulatory Alignment > Stability > Actionability
```

If rankings remain stable across schemes, the EES conclusions are more robust and less dependent on any one weighting assumption.

In [15]:
# =========================================================
# 14. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_06 EES AHP Score completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("Key outputs:")
print("- ees_ahp_score_summary.csv")
print("- ees_final_model_scores.csv")
print("- ees_component_scores_input.csv")
print("- ees_component_contribution_long.csv")
print("- ees_weighting_sensitivity.csv")
print("- ees_ranking_robustness_summary.csv")
print("- ees_ahp_scheme_weights.csv")
print("- ees_ahp_scheme_consistency_diagnostics.csv")
print(f"- ees_ahp_score_{DATASET_NAME}_outputs.xlsx")

08_06 EES AHP Score completed successfully for: COMPAS
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/compas/08_06_ees_framework_ees_ahp_score_compas/logs
Key outputs:
- ees_ahp_score_summary.csv
- ees_final_model_scores.csv
- ees_component_scores_input.csv
- ees_component_contribution_long.csv
- ees_weighting_sensitivity.csv
- ees_ranking_robustness_summary.csv
- ees_ahp_scheme_weights.csv
- ees_ahp_scheme_consistency_diagnostics.csv
- ees_ahp_score_compas_outputs.xlsx
